### Part A - Section 2: Cleaning Pipeline.

### Objective Of This Section.
Creating a reusable function `clean_ops_data(df)` that performs comprehensive data cleaning on the raw sensor dataset.

### Key Data Quality Issues Addressed Here.
1. **Timestamp Format** - Converted from string to proper `datetime` objects for time-based analysis.
2. **Inconsistent Categorical Data**:
- Location names had multiple variations (`mombasa`, `Mombasa`, `MBA`, `NRB`, `Nairobi`).
- Standardized to consistent values: `Mombasa`, `Nairobi`, `Kisumu`.
3. **Missing Values** - Although none were present, implemented forward-fill strategy suitable for time-series data.
4. **Duplicates** - Removed exact duplicate records based on timestamp, sensor, and location.
5. **Invalid Sensor Readings** - Filtered out physically impossible values (outside 0–100 range for temperature/pressure sensors).

In [ ]:
# Cleaning Operational Sensor Data For Analysis From Messy Data.

import pandas as pd

def clean_ops_data(df):
    """
    Creating a Reusable function to clean the operational sensor data.
    """
    df = df.copy()
    
    # 1. Converting timestamp to datetime.
    df['timestamp'] = pd.to_datetime(df['timestamp'], 
                                     format='%d/%m/%Y %H:%M', 
                                     errors='coerce')
    
    # 2. Standardizing the Location names.
    location_map = {
        'mombasa': 'Mombasa',
        'Mombasa': 'Mombasa',
        'MBA': 'Mombasa',
        'NRB': 'Nairobi',
        'Nairobi': 'Nairobi',
        'Kisumu': 'Kisumu'
    }
    df['location'] = df['location'].str.strip().map(location_map)
    
    # 3. Handling the missing values - FIXED for the new pandas.
    df['reading'] = df['reading'].ffill()      # This is the correct way now
    df = df.dropna(subset=['timestamp', 'reading'])
    
    # 4. Removing the duplicates from the raw data.
    df = df.drop_duplicates(subset=['timestamp', 'sensor_id', 'location'])
    
    # 5. Filtering the physically impossible readings.
    df = df[(df['reading'] >= 0) & (df['reading'] <= 100)]
    
    # 6. Standardizing the sensor_id.
    sensor_map = {
        'temp_sensor': 'Temperature_Sensor',
        'TEMP_SENS': 'Temperature_Sensor',
        'Pressure_01': 'Pressure_01',
        'P1': 'Pressure_01'
    }
    df['sensor_id'] = df['sensor_id'].map(sensor_map).fillna(df['sensor_id'])
    
    return df

In [ ]:
#Testing the function with the provided dataset
raw_df = pd.read_csv(r'C:\Users\JetCore Computers\Desktop\PROJECTS\week2_sensor_readings.csv')

cleaned_df = clean_ops_data(raw_df)

print(f"Original dataset shape : {raw_df.shape}")
print(f"Cleaned dataset shape  : {cleaned_df.shape}")
print("\nUnique Locations after cleaning:", sorted(cleaned_df['location'].unique()))
print("Missing values after cleaning:", cleaned_df.isnull().sum().sum())
print("\nSample of cleaned data:")
print(cleaned_df.head(3))